# Wikipedia Pets MiniLM Embeddings

This notebook builds semantic-search-ready chunk metadata and CPU-friendly embeddings for the cleaned Wikipedia pets corpus using `sentence-transformers/all-MiniLM-L6-v2` on CPU.

The pipeline writes chunk metadata and embedding parts to disk so it stays practical on a Windows machine without CUDA or GPU acceleration.

## Dependency Notes

You may need these packages before running the notebook:

- `sentence-transformers`
- `spacy`
- `pandas`
- `pyarrow`
- `numpy`

Example install command:

```bash
pip install sentence-transformers spacy pandas pyarrow numpy
```

This notebook uses `spacy.blank('en')` with the sentencizer, so it does not require downloading `en_core_web_sm`.

In [15]:
from pathlib import Path
import gc
import json
import time

import numpy as np
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer

In [16]:
REPO_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(
    (
        path
        for path in REPO_ROOT_CANDIDATES
        if (path / 'data').exists() or (path / 'output').exists()
    ),
    Path.cwd().resolve(),
)

CORPUS_DIR = REPO_ROOT / 'data' / 'corpus_cleaned'
PAGE_INDEX_FILE = REPO_ROOT / 'output' / 'wikipedia-pets' / 'page_index.json'
EMBEDDINGS_DIR = REPO_ROOT / 'data' / 'embeddings'
METADATA_PARTS_DIR = EMBEDDINGS_DIR / 'chunk_metadata_parts'
EMBEDDING_PARTS_DIR = EMBEDDINGS_DIR / 'embedding_parts'
FINAL_METADATA_FILE = EMBEDDINGS_DIR / 'chunk_metadata.parquet'
FINAL_EMBEDDINGS_FILE = EMBEDDINGS_DIR / 'chunk_embeddings.npy'
SKIPPED_PAGES_FILE = METADATA_PARTS_DIR / 'skipped_pages.parquet'

for directory in (EMBEDDINGS_DIR, METADATA_PARTS_DIR, EMBEDDING_PARTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

NLP = spacy.blank('en')
if 'sentencizer' not in NLP.pipe_names:
    NLP.add_pipe('sentencizer')
NLP.max_length = max(NLP.max_length, 5_000_000)

print(f'Repo root: {REPO_ROOT}')
print(f'Corpus dir: {CORPUS_DIR}')
print(f'Page index file: {PAGE_INDEX_FILE}')

Repo root: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization
Corpus dir: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\corpus_cleaned
Page index file: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\output\wikipedia-pets\page_index.json


In [17]:
def read_file_contents(file_path):
    file_path = Path(file_path)
    with file_path.open('r', encoding='utf-8-sig', errors='replace') as handle:
        return handle.read()


def _short_preview(text, limit=120):
    normalized = ' '.join(text.split())
    if len(normalized) <= limit:
        return normalized
    return normalized[: limit - 3] + '...'


def smart_text_chunking(text, chunk_size=180):
    if chunk_size <= 0:
        raise ValueError('chunk_size must be a positive integer')

    sentences = [sent.text.strip() for sent in NLP(text).sents if sent.text.strip()]
    if not sentences:
        return []

    chunks = []
    current_sentences = []
    current_word_count = 0

    for sentence in sentences:
        sentence_word_count = len(sentence.split())
        if sentence_word_count > chunk_size:
            raise ValueError(
                f'Sentence exceeds chunk_size={chunk_size}: {_short_preview(sentence)}'
            )

        if current_sentences and current_word_count + sentence_word_count > chunk_size:
            chunk = ' '.join(current_sentences).strip()
            if chunk:
                chunks.append(chunk)
            current_sentences = [sentence]
            current_word_count = sentence_word_count
        else:
            current_sentences.append(sentence)
            current_word_count += sentence_word_count

    if current_sentences:
        chunk = ' '.join(current_sentences).strip()
        if chunk:
            chunks.append(chunk)

    return chunks

In [18]:
def _metadata_part_path(part_index):
    return METADATA_PARTS_DIR / f'chunk_metadata_part_{part_index:04d}.parquet'


def _embedding_part_path(metadata_part_path):
    return EMBEDDING_PARTS_DIR / metadata_part_path.name.replace(
        'chunk_metadata', 'chunk_embeddings'
    ).replace('.parquet', '.npy')


def _remove_existing_files(paths):
    for path in paths:
        if path.exists():
            path.unlink()


def build_chunk_metadata_batched(chunk_size=180, batch_pages=500, overwrite=False):
    if batch_pages <= 0:
        raise ValueError('batch_pages must be a positive integer')
    if not PAGE_INDEX_FILE.exists():
        raise FileNotFoundError(f'Page index file not found: {PAGE_INDEX_FILE}')
    if not CORPUS_DIR.exists():
        raise FileNotFoundError(f'Corpus directory not found: {CORPUS_DIR}')

    existing_parts = sorted(METADATA_PARTS_DIR.glob('chunk_metadata_part_*.parquet'))
    if existing_parts and not overwrite:
        print(
            f'Found {len(existing_parts)} existing metadata part file(s) in {METADATA_PARTS_DIR}. '
            'Set overwrite=True to rebuild them.'
        )
        return existing_parts

    if overwrite:
        _remove_existing_files(existing_parts)
        _remove_existing_files([SKIPPED_PAGES_FILE, FINAL_METADATA_FILE])

    page_index = json.loads(read_file_contents(PAGE_INDEX_FILE))
    total_pages = len(page_index)
    skipped_pages = []
    metadata_rows = []
    global_chunk_id = 0
    part_index = 0
    processed_pages = 0
    start_time = time.time()

    def flush_rows(current_part_index):
        if not metadata_rows:
            return current_part_index

        part_path = _metadata_part_path(current_part_index)
        part_df = pd.DataFrame(metadata_rows)
        part_df.to_parquet(part_path, index=False)
        print(f'Saved {part_path.name} with {len(part_df):,} chunk row(s).')
        metadata_rows.clear()
        return current_part_index + 1

    for entry in page_index:
        processed_pages += 1
        page_id = entry.get('page_id')
        title = entry.get('title')
        text_file = entry.get('text_file')

        if not text_file:
            skipped_pages.append(
                {
                    'page_id': page_id,
                    'title': title,
                    'file_name': None,
                    'reason': 'missing text_file',
                }
            )
        else:
            file_name = text_file[6:] if text_file.startswith('texts/') else text_file
            article_path = CORPUS_DIR / file_name

            if not article_path.exists():
                skipped_pages.append(
                    {
                        'page_id': page_id,
                        'title': title,
                        'file_name': file_name,
                        'reason': 'missing corpus file',
                    }
                )
            else:
                try:
                    article_text = read_file_contents(article_path)
                    chunks = smart_text_chunking(article_text, chunk_size=chunk_size)
                    if not chunks:
                        skipped_pages.append(
                            {
                                'page_id': page_id,
                                'title': title,
                                'file_name': file_name,
                                'reason': 'empty text after chunking',
                            }
                        )
                    else:
                        for chunk_index, chunk in enumerate(chunks):
                            metadata_rows.append(
                                {
                                    'global_chunk_id': global_chunk_id,
                                    'page_id': page_id,
                                    'title': title,
                                    'file_name': file_name,
                                    'chunk_index': chunk_index,
                                    'word_count': len(chunk.split()),
                                    'char_count': len(chunk),
                                    'text': chunk,
                                }
                            )
                            global_chunk_id += 1
                except Exception as exc:
                    skipped_pages.append(
                        {
                            'page_id': page_id,
                            'title': title,
                            'file_name': file_name,
                            'reason': f'{type(exc).__name__}: {exc}',
                        }
                    )

        if processed_pages % batch_pages == 0:
            part_index = flush_rows(part_index)

        if processed_pages % 100 == 0 or processed_pages == total_pages:
            elapsed = time.time() - start_time
            print(f'Processed {processed_pages:,}/{total_pages:,} pages in {elapsed:.1f}s')

    part_index = flush_rows(part_index)

    skipped_df = pd.DataFrame(
        skipped_pages,
        columns=['page_id', 'title', 'file_name', 'reason'],
    )
    skipped_df.to_parquet(SKIPPED_PAGES_FILE, index=False)

    print(f'Saved skipped pages report to {SKIPPED_PAGES_FILE}')
    print(f'Finished metadata build with {global_chunk_id:,} chunk(s) across {part_index} part file(s).')
    print(f'Skipped {len(skipped_df):,} page(s).')

    return sorted(METADATA_PARTS_DIR.glob('chunk_metadata_part_*.parquet'))

In [19]:
def combine_metadata_parts():
    metadata_part_files = sorted(METADATA_PARTS_DIR.glob('chunk_metadata_part_*.parquet'))
    if not metadata_part_files:
        raise FileNotFoundError(f'No metadata part files found in {METADATA_PARTS_DIR}')

    frames = [pd.read_parquet(path) for path in metadata_part_files]
    combined_df = pd.concat(frames, ignore_index=True)
    combined_df = combined_df.sort_values('global_chunk_id').reset_index(drop=True)
    combined_df.to_parquet(FINAL_METADATA_FILE, index=False)

    print(
        f'Saved combined metadata to {FINAL_METADATA_FILE} with {len(combined_df):,} row(s).'
    )
    return combined_df


def encode_metadata_parts(batch_size=64, overwrite=False):
    if batch_size <= 0:
        raise ValueError('batch_size must be a positive integer')

    metadata_part_files = sorted(METADATA_PARTS_DIR.glob('chunk_metadata_part_*.parquet'))
    if not metadata_part_files:
        raise FileNotFoundError(f'No metadata part files found in {METADATA_PARTS_DIR}')

    if overwrite:
        _remove_existing_files(sorted(EMBEDDING_PARTS_DIR.glob('chunk_embeddings_part_*.npy')))
        _remove_existing_files([FINAL_EMBEDDINGS_FILE])

    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
    written_files = []

    for metadata_part_path in metadata_part_files:
        embedding_part_path = _embedding_part_path(metadata_part_path)
        if embedding_part_path.exists() and not overwrite:
            print(f'Skipping existing embedding part: {embedding_part_path.name}')
            written_files.append(embedding_part_path)
            continue

        part_start = time.time()
        metadata_df = pd.read_parquet(metadata_part_path)
        metadata_df = metadata_df.sort_values('global_chunk_id').reset_index(drop=True)

        if metadata_df.empty:
            embeddings = np.empty((0, 384), dtype=np.float32)
        else:
            texts = metadata_df['text'].fillna('').astype(str).tolist()
            embeddings = model.encode(
                texts,
                batch_size=batch_size,
                normalize_embeddings=True,
                show_progress_bar=True,
                convert_to_numpy=True,
            )
            embeddings = embeddings.astype(np.float32, copy=False)
            del texts

        np.save(embedding_part_path, embeddings)
        elapsed = time.time() - part_start
        print(
            f'Saved {embedding_part_path.name} with shape {embeddings.shape} in {elapsed:.1f}s'
        )

        written_files.append(embedding_part_path)
        del metadata_df, embeddings
        gc.collect()

    return written_files


def combine_embedding_parts():
    embedding_part_files = sorted(EMBEDDING_PARTS_DIR.glob('chunk_embeddings_part_*.npy'))
    if not embedding_part_files:
        raise FileNotFoundError(f'No embedding part files found in {EMBEDDING_PARTS_DIR}')

    print(
        'Combining embedding parts can use significant RAM. Keep the part files if one large array is too heavy for memory.'
    )
    arrays = [np.load(path) for path in embedding_part_files]
    combined_embeddings = np.vstack(arrays).astype(np.float32, copy=False)
    np.save(FINAL_EMBEDDINGS_FILE, combined_embeddings)

    print(
        f'Saved combined embeddings to {FINAL_EMBEDDINGS_FILE} with shape {combined_embeddings.shape}.'
    )
    return combined_embeddings


def validate_outputs():
    metadata_part_files = sorted(METADATA_PARTS_DIR.glob('chunk_metadata_part_*.parquet'))
    embedding_part_files = sorted(EMBEDDING_PARTS_DIR.glob('chunk_embeddings_part_*.npy'))

    if not metadata_part_files:
        raise FileNotFoundError(f'No metadata part files found in {METADATA_PARTS_DIR}')
    if not embedding_part_files:
        raise FileNotFoundError(f'No embedding part files found in {EMBEDDING_PARTS_DIR}')

    total_metadata_rows = 0
    for metadata_part_path in metadata_part_files:
        total_metadata_rows += len(pd.read_parquet(metadata_part_path, columns=['global_chunk_id']))

    total_embedding_rows = 0
    embedding_dims = []
    for embedding_part_path in embedding_part_files:
        embeddings = np.load(embedding_part_path)
        if embeddings.ndim != 2:
            raise ValueError(
                f'Expected a 2D embedding array in {embedding_part_path}, got shape {embeddings.shape}'
            )
        total_embedding_rows += embeddings.shape[0]
        embedding_dims.append(embeddings.shape[1])
        del embeddings

    counts_match = total_metadata_rows == total_embedding_rows
    dims_match = all(dim == 384 for dim in embedding_dims)

    print('Validation summary')
    print(f'- metadata part files: {len(metadata_part_files)}')
    print(f'- embedding part files: {len(embedding_part_files)}')
    print(f'- metadata rows: {total_metadata_rows:,}')
    print(f'- embedding rows: {total_embedding_rows:,}')
    print(f'- embedding dimensions: {sorted(set(embedding_dims))}')
    print(f'- combined metadata present: {FINAL_METADATA_FILE.exists()}')
    print(f'- combined embeddings present: {FINAL_EMBEDDINGS_FILE.exists()}')

    if not counts_match:
        raise ValueError('Metadata row count does not match embedding row count.')
    if not dims_match:
        raise ValueError('Expected all-MiniLM-L6-v2 embedding dimensions to be 384.')

    print('Validation passed.')
    return {
        'metadata_rows': total_metadata_rows,
        'embedding_rows': total_embedding_rows,
        'embedding_dimensions': sorted(set(embedding_dims)),
        'combined_metadata_exists': FINAL_METADATA_FILE.exists(),
        'combined_embeddings_exists': FINAL_EMBEDDINGS_FILE.exists(),
    }


def _score_document_from_chunk_scores(chunk_scores):
    ranked_chunk_scores = np.sort(np.asarray(chunk_scores, dtype=np.float32))[::-1]
    if ranked_chunk_scores.size == 0:
        return 0.0

    weighted_score = 0.0
    used_weight = 0.0
    top_chunk_weights = np.array([0.40, 0.35, 0.20], dtype=np.float32)
    top_chunk_count = min(3, ranked_chunk_scores.size)

    if top_chunk_count:
        weighted_score += float(
            np.dot(ranked_chunk_scores[:top_chunk_count], top_chunk_weights[:top_chunk_count])
        )
        used_weight += float(top_chunk_weights[:top_chunk_count].sum())

    if ranked_chunk_scores.size > 3:
        weighted_score += 0.05 * float(ranked_chunk_scores[3:].mean())
        used_weight += 0.05

    return weighted_score / used_weight if used_weight else 0.0


def _document_score_breakdown(chunk_scores):
    ranked_chunk_scores = np.sort(np.asarray(chunk_scores, dtype=np.float32))[::-1]
    return {
        'top_chunk_score_1': float(ranked_chunk_scores[0]) if ranked_chunk_scores.size >= 1 else np.nan,
        'top_chunk_score_2': float(ranked_chunk_scores[1]) if ranked_chunk_scores.size >= 2 else np.nan,
        'top_chunk_score_3': float(ranked_chunk_scores[2]) if ranked_chunk_scores.size >= 3 else np.nan,
        'remaining_chunks_mean_score': float(ranked_chunk_scores[3:].mean()) if ranked_chunk_scores.size > 3 else np.nan,
    }


def _load_full_document_text(file_name, fallback_chunks):
    if file_name:
        document_path = CORPUS_DIR / file_name
        if document_path.exists():
            return read_file_contents(document_path).strip()

    return '\n\n'.join(
        chunk.strip() for chunk in fallback_chunks if isinstance(chunk, str) and chunk.strip()
    )


def semantic_search(query, top_k=5):
    if top_k <= 0:
        raise ValueError('top_k must be a positive integer')
    if not FINAL_METADATA_FILE.exists():
        raise FileNotFoundError(
            f'Combined metadata file not found: {FINAL_METADATA_FILE}. Run combine_metadata_parts() first.'
        )
    if not FINAL_EMBEDDINGS_FILE.exists():
        raise FileNotFoundError(
            f'Combined embeddings file not found: {FINAL_EMBEDDINGS_FILE}. Run combine_embedding_parts() first.'
        )

    metadata_df = pd.read_parquet(FINAL_METADATA_FILE)
    metadata_df = metadata_df.sort_values('global_chunk_id').reset_index(drop=True)
    embeddings = np.load(FINAL_EMBEDDINGS_FILE)

    if embeddings.ndim != 2:
        raise ValueError(f'Expected a 2D embedding array, got shape {embeddings.shape}')
    if len(metadata_df) != embeddings.shape[0]:
        raise ValueError('Combined metadata and embedding row counts do not match.')

    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).astype(np.float32, copy=False)[0]

    chunk_scores = embeddings @ query_embedding
    if chunk_scores.size == 0:
        return pd.DataFrame(
            columns=[
                'score',
                'page_id',
                'title',
                'file_name',
                'chunk_count',
                'top_chunk_score_1',
                'top_chunk_score_2',
                'top_chunk_score_3',
                'remaining_chunks_mean_score',
                'document_text',
            ]
        )

    search_df = metadata_df[['page_id', 'title', 'file_name', 'chunk_index', 'text']].copy()
    search_df['chunk_score'] = chunk_scores.astype(np.float32, copy=False)

    document_scores = (
        search_df.groupby(['page_id', 'title', 'file_name'], sort=False, dropna=False)['chunk_score']
        .apply(_score_document_from_chunk_scores)
        .reset_index(name='score')
    )

    if document_scores.empty:
        return pd.DataFrame(
            columns=[
                'score',
                'page_id',
                'title',
                'file_name',
                'chunk_count',
                'top_chunk_score_1',
                'top_chunk_score_2',
                'top_chunk_score_3',
                'remaining_chunks_mean_score',
                'document_text',
            ]
        )

    document_score_values = document_scores['score'].to_numpy(dtype=np.float32, copy=False)
    top_k = min(top_k, document_score_values.shape[0])

    if top_k == document_score_values.shape[0]:
        candidate_indices = np.arange(document_score_values.shape[0])
    else:
        candidate_indices = np.argpartition(document_score_values, -top_k)[-top_k:]

    ranked_indices = candidate_indices[np.argsort(document_score_values[candidate_indices])[::-1]]
    top_documents = document_scores.iloc[ranked_indices].sort_values(
        ['score', 'title'],
        ascending=[False, True],
    )

    results = []
    for row in top_documents.itertuples(index=False):
        document_chunks = search_df[search_df['page_id'] == row.page_id].sort_values('chunk_index')
        score_breakdown = _document_score_breakdown(document_chunks['chunk_score'].to_numpy())
        results.append(
            {
                'score': float(row.score),
                'page_id': row.page_id,
                'title': row.title,
                'file_name': row.file_name,
                'chunk_count': int(len(document_chunks)),
                'top_chunk_score_1': score_breakdown['top_chunk_score_1'],
                'top_chunk_score_2': score_breakdown['top_chunk_score_2'],
                'top_chunk_score_3': score_breakdown['top_chunk_score_3'],
                'remaining_chunks_mean_score': score_breakdown['remaining_chunks_mean_score'],
                'document_text': _load_full_document_text(
                    row.file_name,
                    document_chunks['text'].tolist(),
                ),
            }
        )

    return pd.DataFrame(results).reset_index(drop=True)

## Practical Notes

- Start with `batch_size=64`.
- If RAM goes above 90%, reduce `batch_size` to `32`.
- If RAM stays comfortable, try `batch_size=128`.
- Do not use multiprocessing initially because it may load multiple model copies and increase RAM usage.
- `all-MiniLM-L6-v2` creates 384-dimensional embeddings.
- Because `normalize_embeddings=True` is used, dot product is equivalent to cosine similarity.
- The semantic-search demo now ranks whole documents, not individual chunks. Document scores use 40% from the best chunk, 35% from the second, 20% from the third, and 5% from the mean score of the remaining chunks.
- Keep metadata and embedding part files if the full combined files are too large for RAM.

## Execution Order

In [7]:
build_chunk_metadata_batched(chunk_size=180, batch_pages=500, overwrite=False)

Processed 100/21,077 pages in 1.3s
Processed 200/21,077 pages in 1.7s
Processed 300/21,077 pages in 2.2s
Processed 400/21,077 pages in 3.1s
Saved chunk_metadata_part_0000.parquet with 2,789 chunk row(s).
Processed 500/21,077 pages in 4.4s
Processed 600/21,077 pages in 4.9s
Processed 700/21,077 pages in 5.8s
Processed 800/21,077 pages in 7.0s
Processed 900/21,077 pages in 8.3s
Saved chunk_metadata_part_0001.parquet with 2,118 chunk row(s).
Processed 1,000/21,077 pages in 9.1s
Processed 1,100/21,077 pages in 11.6s
Processed 1,200/21,077 pages in 13.3s
Processed 1,300/21,077 pages in 15.8s
Processed 1,400/21,077 pages in 17.0s
Saved chunk_metadata_part_0002.parquet with 4,737 chunk row(s).
Processed 1,500/21,077 pages in 18.3s
Processed 1,600/21,077 pages in 19.8s
Processed 1,700/21,077 pages in 21.6s
Processed 1,800/21,077 pages in 23.9s
Processed 1,900/21,077 pages in 25.1s
Saved chunk_metadata_part_0003.parquet with 4,768 chunk row(s).
Processed 2,000/21,077 pages in 26.9s
Processed 2,

[WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0000.parquet'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0001.parquet'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0002.parquet'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0003.parquet'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0004.parquet'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0005.parquet'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_m

In [8]:
metadata_part_files = sorted(METADATA_PARTS_DIR.glob('chunk_metadata_part_*.parquet'))
print(f'Found {len(metadata_part_files)} metadata part file(s).')
if not metadata_part_files:
    raise FileNotFoundError(f'No metadata parts found in {METADATA_PARTS_DIR}')
metadata_part_files[:3]

Found 43 metadata part file(s).


[WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0000.parquet'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0001.parquet'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/chunk_metadata_parts/chunk_metadata_part_0002.parquet')]

In [9]:
encode_metadata_parts(batch_size=64, overwrite=False)

Batches: 100%|██████████| 44/44 [01:08<00:00,  1.55s/it]


Saved chunk_embeddings_part_0000.npy with shape (2789, 384) in 69.0s


Batches: 100%|██████████| 34/34 [00:49<00:00,  1.45s/it]


Saved chunk_embeddings_part_0001.npy with shape (2118, 384) in 49.3s


Batches: 100%|██████████| 75/75 [01:55<00:00,  1.55s/it]


...

Batches: 100%|██████████| 84/84 [02:16<00:00,  1.62s/it]


Saved chunk_embeddings_part_0041.npy with shape (5345, 384) in 136.4s


Batches: 100%|██████████| 4/4 [00:04<00:00,  1.11s/it]


Saved chunk_embeddings_part_0042.npy with shape (253, 384) in 4.6s


[WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/embedding_parts/chunk_embeddings_part_0000.npy'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/embedding_parts/chunk_embeddings_part_0001.npy'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/embedding_parts/chunk_embeddings_part_0002.npy'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/embedding_parts/chunk_embeddings_part_0003.npy'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/embedding_parts/chunk_embeddings_part_0004.npy'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/embedding_parts/chunk_embeddings_part_0005.npy'),
 WindowsPath('D:/University/FCSE/Courses/26S/TUG-26S-BT-Content_Optimization/data/embeddings/embedding_parts/chunk_embeddings_part_0006.npy'),

In [20]:
validate_outputs()

Validation summary
- metadata part files: 43
- embedding part files: 43
- metadata rows: 164,651
- embedding rows: 164,651
- embedding dimensions: [384]
- combined metadata present: True
- combined embeddings present: True
Validation passed.


{'metadata_rows': 164651,
 'embedding_rows': 164651,
 'embedding_dimensions': [384],
 'combined_metadata_exists': True,
 'combined_embeddings_exists': True}

In [ ]:
combine_metadata_parts()

Saved combined metadata to D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\embeddings\chunk_metadata.parquet with 164,651 row(s).


,global_chunk_id,page_id,title,file_name,chunk_index,word_count,char_count,text
0,0,7590101,(Blooper) Bunny,(Blooper)_Bunny_8dbe4cbf9c.txt,0,171,996,(Blooper) Bunny is a Merrie Melodies animated ...
1,1,7590101,(Blooper) Bunny,(Blooper)_Bunny_8dbe4cbf9c.txt,1,174,1019,Elmer Fudd is shown trying to use minoxidil to...
2,2,7590101,(Blooper) Bunny,(Blooper)_Bunny_8dbe4cbf9c.txt,2,163,890,"While waiting for the cane to be thrown, it be..."
3,3,7590101,(Blooper) Bunny,(Blooper)_Bunny_8dbe4cbf9c.txt,3,162,833,"Bugs scolds him, but Elmer responds that he th..."
4,4,7590101,(Blooper) Bunny,(Blooper)_Bunny_8dbe4cbf9c.txt,4,169,977,Everything plays out correctly until Yosemite ...
...,...,...,...,...,...,...,...,...
164646,164646,72384616,ワッカネズミ,ワッカネズミ_78df9df5a7.txt,1,170,1011,"Each Pokémon have one or two elemental types, ..."
164647,164647,72384616,ワッカネズミ,ワッカネズミ_78df9df5a7.txt,2,42,246,"It also saw the return of the "" Mega Evolution..."
164648,164648,72384535,ワナイダー,ワナイダー_7493e3fb20.txt,0,179,1093,"""Flittle"" redirects here. For the Disney chara..."
164649,164649,72384535,ワナイダー,ワナイダー_7493e3fb20.txt,1,170,1011,"Each Pokémon have one or two elemental types, ..."


In [ ]:
combine_embedding_parts()

Combining embedding parts can use significant RAM. Keep the part files if one large array is too heavy for memory.
Saved combined embeddings to D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\embeddings\chunk_embeddings.npy with shape (164651, 384).


array([[ 0.08009247,  0.03471058,  0.0647997 , ...,  0.01337484,
         0.04426945, -0.01809438],
       [-0.04752256,  0.01847462, -0.01661636, ...,  0.01823028,
         0.06246856,  0.08309358],
       [-0.06954007, -0.02972421,  0.00265219, ..., -0.01711483,
         0.08719522,  0.08415226],
       ...,
       [-0.09890335, -0.04979947,  0.02612593, ...,  0.00550245,
         0.06005381,  0.00975198],
       [-0.01707192, -0.02661807,  0.00755279, ..., -0.03183546,
        -0.0244628 ,  0.00717749],
       [-0.12808706,  0.04095592,  0.04956165, ...,  0.02536603,
         0.05142136,  0.05133188]], shape=(164651, 384), dtype=float32)

In [21]:
if FINAL_METADATA_FILE.exists() and FINAL_EMBEDDINGS_FILE.exists():
    demo_results = semantic_search('small dogs that are good with children', top_k=5)
    demo_results
else:
    print('Combined files not found. Run combine_metadata_parts() and combine_embedding_parts() first.')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5093.48it/s]


In [22]:
demo_results

,score,page_id,title,file_name,chunk_count,top_chunk_score_1,top_chunk_score_2,top_chunk_score_3,remaining_chunks_mean_score,document_text
0,0.544222,727149,Miniature Fox Terrier,Miniature_Fox_Terrier_8592b652e6.txt,8,0.577976,0.552426,0.490193,0.432886,Miniature Fox Terrier\n\n( Learn how and when ...
1,0.527607,731386,Toy dog,Toy_dog_0f77d06b3b.txt,5,0.559858,0.547644,0.452425,0.430061,Not to be confused with Dog toy .\n\nToy dog t...
2,0.518437,790060,Lap dog,Lap_dog_58dd16af8b.txt,9,0.531919,0.521450,0.518338,0.389895,"""lap pet"" redirects here. For the garment, see..."
3,0.501081,14418001,List of dog crossbreeds,List_of_dog_crossbreeds_06a4f10530.txt,7,0.545928,0.475895,0.465711,0.460098,This is a list of common dog crossbreeds . The...
4,0.500204,52069460,Cão de Gado Transmontano,Cão_de_Gado_Transmontano_e98e7a4ee9.txt,7,0.536145,0.496032,0.476671,0.336013,Cão de Gado Transmontano\n\nDog ( domestic dog...
